In [ ]:
import warnings
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
import xgboost as xgb
import yfinance as yf

warnings.filterwarnings('ignore')

print(
    '--- [BANCADA DE MACHINE LEARNING]: INICIANDO PIPELINE NÃO-LINEAR XGBOOST'
    ' COMPLETO ---'
)

# ==============================================================================
# 1. CARGA E TRATAMENTO DE DADOS LOCAIS
# ==============================================================================
df_credito = pd.read_csv(
    'bacen_credito_spread_inadimplencia.csv', sep=';', encoding='latin1'
)
for col in df_credito.columns[1:]:
  df_credito[col] = (
      df_credito[col].astype(str).str.replace(',', '.').astype(float)
  )

meses_map = {
    'jan': '01',
    'fev': '02',
    'mar': '03',
    'abr': '04',
    'mai': '05',
    'jun': '06',
    'jul': '07',
    'ago': '08',
    'set': '09',
    'out': '10',
    'nov': '11',
    'dez': '12',
}


def parse_sgs_date(date_str):
  mes, ano = date_str.split('/')
  return f'20{ano}-{meses_map[mes]}-01'


df_credito['Data_Merge'] = pd.to_datetime(
    df_credito['Data'].apply(parse_sgs_date)
)

df_selic_diaria = pd.read_csv(
    'bacen_taxa_selic_diaria.csv', sep=';', encoding='latin1'
)
df_selic_diaria['Data'] = pd.to_datetime(
    df_selic_diaria['Data'], format='%d/%m/%Y'
)
df_selic_diaria.columns = ['Data', 'Selic']
df_selic_diaria['Selic'] = (
    df_selic_diaria['Selic'].astype(str).str.replace(',', '.').astype(float)
)
df_selic_mensal = (
    df_selic_diaria.resample('MS', on='Data').mean().reset_index()
)
df_selic_mensal.columns = ['Data_Merge', 'Selic_Over']

df_target = pd.read_excel('fgv_ice_original.xls', sheet_name='sheet')
df_target['Data_Merge'] = pd.to_datetime(
    df_target['Data']
    .str.split('/')
    .apply(lambda x: f'{x[1]}-{x[0]}-01')
)
df_target.rename(columns={'ICE': 'ICE'}, inplace=True)


def tratar_investing_diario(arquivo, nome_coluna):
  df = pd.read_csv(arquivo, encoding='utf-8')
  df['Data_dt'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')
  df['Último'] = (
      df['Último']
      .astype(str)
      .str.replace('.', '', regex=False)
      .str.replace(',', '.', regex=False)
      .astype(float)
  )
  df_m = df.resample('MS', on='Data_dt').last().reset_index()
  return df_m[['Data_dt', 'Último']].rename(
      columns={'Data_dt': 'Data_Merge', 'Último': nome_coluna}
  )


df_cds = tratar_investing_diario(
    'investing_cds_5y_brasil.csv', 'CDS_5Y_Brasil'
)
df_longo = tratar_investing_diario(
    'investing_juro_longo_5y.csv', 'Juro_Longo_5Y'
)
df_curto = tratar_investing_diario(
    'investing_juro_curto_1y.csv', 'Juro_Curto_1Y'
)

df_curva = pd.merge(df_longo, df_curto, on='Data_Merge', how='inner')
df_curva['Inclinacao_Curva'] = (
    df_curva['Juro_Longo_5Y'] - df_curva['Juro_Curto_1Y']
)

# ==============================================================================
# 2. EXTRAÇÃO VIA API (YAHOO FINANCE) E CONSOLIDAÇÃO
# ==============================================================================
data_inicio, data_fim = '2011-03-01', '2026-06-01'
tickers = {
    'Ibovespa': '^BVSP',
    'VIX_Global': '^VIX',
    'Cambio_USD_BRL': 'BRL=X',
}
df_mercado_mensal = pd.DataFrame()

print('Puxando dados de mercado do Yahoo Finance...')

for nome, ticker in tickers.items():
  data_ticker = yf.download(
      ticker, start=data_inicio, end=data_fim, interval='1d', auto_adjust=True
  )
  serie_close = (
      data_ticker.loc[:, ('Close', ticker)]
      if isinstance(data_ticker.columns, pd.MultiIndex)
      else data_ticker['Close']
  )

  if nome == 'Cambio_USD_BRL':
    daily_ret = serie_close.pct_change()
    vol_mensal = daily_ret.resample('MS').std() * np.sqrt(252) * 100
    df_mercado_mensal['Vol_Implicita_Cambio'] = vol_mensal

  df_mercado_mensal[nome] = serie_close.resample('MS').last()

bvsp_data = yf.download(
    '^BVSP', start=data_inicio, end=data_fim, interval='1d'
)
df_mercado_mensal['Volume_B3'] = (
    bvsp_data.loc[:, ('Volume', '^BVSP')].resample('MS').sum()
    if isinstance(bvsp_data.columns, pd.MultiIndex)
    else bvsp_data['Volume'].resample('MS').sum()
)

df_mercado_mensal = df_mercado_mensal.reset_index()
df_mercado_mensal = df_mercado_mensal.rename(
    columns={'Date': 'Data_Merge', 'Data': 'Data_Merge', 'index': 'Data_Merge'}
)
df_mercado_mensal['Ibovespa_Retorno'] = (
    df_mercado_mensal['Ibovespa'].pct_change() * 100
)

df_final = pd.merge(df_credito, df_selic_mensal, on='Data_Merge', how='inner')
df_final = pd.merge(
    df_final, df_target[['Data_Merge', 'ICE']], on='Data_Merge', how='inner'
)
df_final = pd.merge(df_final, df_cds, on='Data_Merge', how='inner')
df_final = pd.merge(
    df_final,
    df_curva[['Data_Merge', 'Inclinacao_Curva']],
    on='Data_Merge',
    how='inner',
)
df_final = (
    pd.merge(df_final, df_mercado_mensal, on='Data_Merge', how='inner')
    .dropna()
    .reset_index(drop=True)
)

features_raw = [
    c
    for c in df_final.columns
    if c not in ['Data', 'Data_Merge', 'ICE', 'Ibovespa', 'Cambio_USD_BRL']
]

# ==============================================================================
# 3. FILTRO ESTRUTURAL STL
# ==============================================================================
print('Executando Filtro Estrutural STL (period=13)...')

stl_y = STL(df_final['ICE'], period=13, robust=True).fit()
df_final['ICE_SA'] = stl_y.trend + stl_y.resid

for col in features_raw:
  stl_x = STL(df_final[col], period=13, robust=True).fit()
  df_final[col + '_SA'] = stl_x.trend + stl_x.resid

features_sa = [col + '_SA' for col in features_raw]

nome_limpo_map = {
    '20785 - Spread médio das operações de crédito - Pessoas físicas - Total -'
    ' p.p._SA': 'Spread Bancário - Pessoa Física',
    '20787 - Spread médio das operações de crédito com recursos livres -'
    ' Pessoas jurídicas - Total - p.p._SA': 'Spread Bancário - Pessoa Jurídica',
    '21082 - Inadimplência da carteira de crédito - Total - %_SA': (
        'Taxa de Inadimplência do Sistema'
    ),
    'Selic_Over_SA': 'Taxa Selic Over',
    'CDS_5Y_Brasil_SA': 'CDS 5Y Brasil (Risco-País)',
    'Inclinacao_Curva_SA': 'Inclinação da Curva de Juros',
    'VIX_Global_SA': 'Índice VIX Global',
    'Vol_Implicita_Cambio_SA': 'Volatilidade Realizada do Câmbio',
    'Volume_B3_SA': 'Volume Financeiro da B3',
    'Ibovespa_Retorno_SA': 'Retorno Mensal do Ibovespa',
}

# ==============================================================================
# 4. SELEÇÃO DE LAGS ÓTIMOS VIA BIC
# ==============================================================================
print('\n--- DETECTANDO LAGS ÓTIMOS ---')
lags_otimos = {}
X_dinamico_list = []
y_alvo = df_final['ICE_SA']
amostra_comum_idx = df_final.index[12:]

for col in features_sa:
  melhor_lag = 1
  menor_bic = float('inf')

  for lag in range(1, 13):
    feature_lagged = df_final[col].shift(lag)
    X_temp = sm.add_constant(feature_lagged.loc[amostra_comum_idx])
    y_temp = y_alvo.loc[amostra_comum_idx]

    modelo_temp = sm.OLS(y_temp, X_temp).fit()
    bic_atual = modelo_temp.bic

    if bic_atual < menor_bic:
      menor_bic = bic_atual
      melhor_lag = lag

  lags_otimos[col] = melhor_lag
  nome_mercado = nome_limpo_map.get(col, col)
  print(f'-> {nome_mercado:<35} | Lag Ótimo: {melhor_lag} meses')

  serie_otima = df_final[col].shift(melhor_lag)
  serie_otima.name = f'{col}_lag_{melhor_lag}'
  X_dinamico_list.append(serie_otima)

df_X_dinamico = pd.concat(X_dinamico_list, axis=1)
df_valid_dataset = (
    pd.concat([df_final['Data_Merge'], df_X_dinamico, y_alvo], axis=1)
    .dropna()
    .reset_index(drop=True)
)

features_dinamicas_col = [c for c in df_X_dinamico.columns]
X_full = df_valid_dataset[features_dinamicas_col]
y_full = df_valid_dataset['ICE_SA']

# ==============================================================================
# 5. NOMES LIMPOS E DEFINIÇÃO DAS TRAVAS ESTRUTURAIS
# ==============================================================================
nomes_mercado_dinamicos = []
for v in features_dinamicas_col:
  base_name = v.split('_lag_')[0]
  lag_num = v.split('_lag_')[1]
  nome_limpo = nome_limpo_map.get(base_name, base_name)
  nomes_mercado_dinamicos.append(f'{nome_limpo} (t-{lag_num})')

X_full_renamed = X_full.copy()
X_full_renamed.columns = nomes_mercado_dinamicos

# Monotonicidade teórica: -1 (estresse/piora da confiança) | +1 (alívio/melhora)
sinais_monotonicos = {
    'Spread Bancário - Pessoa Física': -1,
    'Spread Bancário - Pessoa Jurídica': -1,
    'Taxa de Inadimplência do Sistema': -1,
    'Taxa Selic Over': -1,
    'CDS 5Y Brasil (Risco-País)': -1,
    'Inclinação da Curva de Juros': 1,
    'Índice VIX Global': -1,
    'Volatilidade Realizada do Câmbio': -1,
    'Volume Financeiro da B3': 1,
    'Retorno Mensal do Ibovespa': 1,
}

restricoes_monotonicas_lista = []
for v in features_dinamicas_col:
  base_name = v.split('_lag_')[0]
  nome_limpo = nome_limpo_map.get(base_name, base_name)
  restricoes_monotonicas_lista.append(sinais_monotonicos.get(nome_limpo, 0))

nomes_bloco_1_clean = []
nomes_bloco_2_clean = []
for v in features_dinamicas_col:
  base_name = v.split('_lag_')[0]
  lag_num = v.split('_lag_')[1]
  nome_limpo = nome_limpo_map.get(base_name, base_name)
  col_renamed = f'{nome_limpo} (t-{lag_num})'
  if nome_limpo in [
      'Taxa Selic Over',
      'CDS 5Y Brasil (Risco-País)',
      'Volatilidade Realizada do Câmbio',
      'Índice VIX Global',
  ]:
    nomes_bloco_1_clean.append(col_renamed)
  else:
    nomes_bloco_2_clean.append(col_renamed)

restricoes_interacao_clean = [nomes_bloco_1_clean, nomes_bloco_2_clean]

# ==============================================================================
# 6. TREINAMENTO, GRID SEARCH E ESTUDO DE ABLAÇÃO
# ==============================================================================
print('\n--- CALIBRANDO HIPERPARÂMETROS VIA GRID SEARCH ---')
X_train_renamed, X_test_renamed, y_train, y_test = train_test_split(
    X_full_renamed, y_full, test_size=0.2, random_state=42
)

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    monotone_constraints=tuple(restricoes_monotonicas_lista),
    interaction_constraints=restricoes_interacao_clean,
    random_state=42,
)
grid_xgb = GridSearchCV(
    xgb_base,
    param_grid_xgb,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
).fit(X_train_renamed, y_train)

print('\n' + '=' * 70)
print('EXECUTANDO ESTUDO DE ABLAÇÃO (CV TEMPORAL EM 5 DOBRAS)')
print('=' * 70)

tscv = TimeSeriesSplit(n_splits=5)


def avaliar_cv_temporal(modelo, X_data, y_data):
  rmse_list, mae_list, r2_list = [], [], []
  for train_idx, val_idx in tscv.split(X_data):
    X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
    y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]

    modelo.fit(X_tr, y_tr)
    preds = modelo.predict(X_val)

    rmse_list.append(np.sqrt(mean_squared_error(y_val, preds)))
    mae_list.append(mean_absolute_error(y_val, preds))
    r2_list.append(r2_score(y_val, preds))

  return np.mean(rmse_list), np.mean(mae_list), np.mean(r2_list)


modelo_irrestrito = xgb.XGBRegressor(**grid_xgb.best_params_, random_state=42)
rmse_uncon, mae_uncon, r2_uncon = avaliar_cv_temporal(
    modelo_irrestrito, X_full_renamed, y_full
)

modelo_restrito = xgb.XGBRegressor(
    **grid_xgb.best_params_,
    monotone_constraints=tuple(restricoes_monotonicas_lista),
    interaction_constraints=restricoes_interacao_clean,
    random_state=42,
)
rmse_con, mae_con, r2_con = avaliar_cv_temporal(
    modelo_restrito, X_full_renamed, y_full
)

df_ablation = pd.DataFrame({
    'Especificação': [
        'XGBoost Irrestrito (Benchmark)',
        'XGBoost com Restrições Estruturais',
    ],
    'RMSE (CV)': [rmse_uncon, rmse_con],
    'MAE (CV)': [mae_uncon, mae_con],
    'R² (CV)': [r2_uncon, r2_con],
})

print(df_ablation.to_string(index=False))
print('=' * 70)

# ==============================================================================
# 7. MODELO FINAL E EXPORTAÇÃO DOS GRÁFICOS
# ==============================================================================
modelo_xgb_clean = xgb.XGBRegressor(
    **grid_xgb.best_params_,
    monotone_constraints=tuple(restricoes_monotonicas_lista),
    interaction_constraints=restricoes_interacao_clean,
    random_state=42,
)
modelo_xgb_clean.fit(X_train_renamed, y_train)

# 7.1. Figura 1: Importância Relativa dos Regressores (Gain)
importances = modelo_xgb_clean.feature_importances_
df_importance = pd.DataFrame(
    {'Feature': nomes_mercado_dinamicos, 'Importance': importances}
).sort_values(by='Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(
    df_importance['Feature'],
    df_importance['Importance'],
    color='navy',
    edgecolor='black',
    alpha=0.8,
)
for bar in bars:
  width = bar.get_width()
  ax.text(
      width + 0.005,
      bar.get_y() + bar.get_height() / 2,
      f'{width:.4f}',
      va='center',
      ha='left',
      fontweight='bold',
      fontsize=9,
  )

ax.set_xlabel('Ganho Relativo (Feature Importance)', fontweight='bold')
ax.set_xlim(0, df_importance['Importance'].max() + 0.05)
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig('top_features_xgboost_pt.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.2. Figura 2: Contribuição Média SHAP com Sinais Econômicos
print('Calculando SHAP e gerando gráfico com sinais calibrados...')
explainer = shap.TreeExplainer(modelo_xgb_clean)
shap_values_cell = explainer(X_test_renamed)

val_shap_matriz = (
    shap_values_cell.values
    if hasattr(shap_values_cell, 'values')
    else shap_values_cell
)
importancia_media_shap = np.mean(np.abs(val_shap_matriz), axis=0)

sinais_impacto = []
for i in range(val_shap_matriz.shape[1]):
  corr = np.corrcoef(X_test_renamed.iloc[:, i], val_shap_matriz[:, i])[0, 1]
  sinais_impacto.append('seagreen' if corr > 0 else 'firebrick')

df_shap_custom = pd.DataFrame({
    'Feature': nomes_mercado_dinamicos,
    'Mean_SHAP': importancia_media_shap,
    'Cor': sinais_impacto,
}).sort_values(by='Mean_SHAP', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars_shap = ax.barh(
    df_shap_custom['Feature'],
    df_shap_custom['Mean_SHAP'],
    color=df_shap_custom['Cor'],
    edgecolor='black',
    alpha=0.85,
)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.2)

for bar in bars_shap:
  width = bar.get_width()
  ax.text(
      width + 0.01,
      bar.get_y() + bar.get_height() / 2,
      f'{width:.4f}',
      va='center',
      ha='left',
      fontweight='bold',
      fontsize=9,
      color='black',
  )

ax.set_xlabel(
    'Impacto Médio Absoluto na Confiança | mean(|SHAP value|)',
    fontweight='bold',
)
ax.set_xlim(0, df_shap_custom['Mean_SHAP'].max() + 0.1)
ax.grid(True, linestyle=':', alpha=0.3)

patch_red = mpatches.Patch(
    color='firebrick', label='Reduz Confiança (Eleva Estresse)'
)
patch_green = mpatches.Patch(
    color='seagreen', label='Eleva Confiança (Atenua Estresse)'
)
ax.legend(
    handles=[patch_red, patch_green],
    loc='lower right',
    frameon=False,
    fontsize=9.5,
)

plt.tight_layout()
plt.savefig('impacto_shap_xgboost_pt.png', dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 8. SÉRIE HISTÓRICA DO VIX TUPINIQUIM II (BASE 100)
# ==============================================================================
df_valid_dataset['XGB_Previsto'] = modelo_xgb_clean.predict(X_full_renamed)

vix_xgb_raw = (
    -1
    * (
        df_valid_dataset['XGB_Previsto']
        - df_valid_dataset['XGB_Previsto'].mean()
    )
    / df_valid_dataset['XGB_Previsto'].std()
)
df_valid_dataset['VIX_Tupiniquim_XGB_Base100'] = 100 + (vix_xgb_raw * 10)

fig, ax = plt.subplots(figsize=(14, 5.2))
recessoes_codace = [('2014-03-01', '2016-12-01'), ('2020-03-01', '2020-06-01')]
for inicio, fim in recessoes_codace:
  ax.axvspan(
      pd.to_datetime(inicio),
      pd.to_datetime(fim),
      color='lightgrey',
      alpha=0.6,
      label='Recessão Oficial CODACE/FGV' if inicio == '2014-03-01' else '',
  )

ax.plot(
    df_valid_dataset['Data_Merge'],
    df_valid_dataset['VIX_Tupiniquim_XGB_Base100'],
    color='darkorange',
    linewidth=2.5,
    label='Índice VIX Tupiniquim (Pipeline XGBoost | Média = 100)',
)
ax.axhline(
    y=100,
    color='black',
    linestyle=':',
    linewidth=1.2,
    label='Normalidade Histórica (Média = 100)',
)

ax.set_xlabel('Anos', fontweight='bold')
ax.set_ylabel('Pontos do Índice (Média = 100)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(loc='best', frameon=False)

plt.tight_layout()
plt.savefig('vix_tupiniquim_base100_xgboost_pt.png', dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 9. MÉTRICAS E EXPORTAÇÃO DOS DADOS
# ==============================================================================
preds_xgb = modelo_xgb_clean.predict(X_test_renamed)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
mae_xgb = mean_absolute_error(y_test, preds_xgb)

print(f'\n-> RMSE (Holdout): {rmse_xgb:.4f} | MAE (Holdout): {mae_xgb:.4f}')

df_export_vix_xgb = df_valid_dataset[
    ['Data_Merge', 'VIX_Tupiniquim_XGB_Base100']
].copy()
df_export_vix_xgb.rename(
    columns={
        'Data_Merge': 'Data',
        'VIX_Tupiniquim_XGB_Base100': 'VIX_Tupiniquim_XGBoost',
    },
    inplace=True,
)
df_export_vix_xgb.to_excel('serie_historica_vix_tupiniquim_ii.xlsx', index=False)
print("[SUCESSO]: 'serie_historica_vix_tupiniquim_ii.xlsx' gravada com sucesso!")